In [1]:
%matplotlib inline
import sys
import os
import time
import numpy as np

import matplotlib.pyplot as plt
import matplotlib as mpl

In [2]:
import os

notebook_path = os.getcwd()
print(notebook_path)

/scratch/ff2183/Causality-Simple-Models/CdV-Good/Physics-constrained/Sequential/Local_Standardization/General_code/Discrete/Test_To_Share/Energy_Conserving


In [3]:
# Mean and standard deviations (the neural networks run in a standardized space)
# We will need this to de-standardize the results back into the real space
mean_i = np.load('./results/train_mean_i.npy')
std_i = np.load('./results/train_std_i.npy')

## Loading the model

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from nn import DiscreteEnergyConservingModel
        
# Initialize the model
input_dim = 6  # Dimensionality of the system
hidden_dim = 100  # Number of hidden units (Note: maybe try hidden_dim > input_dim)
output_dim = 6  # Dimensionality of the output
# Choose the same device as used during training (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Define Dummy Tensors to satisfy the constructor
dummy_A = torch.zeros(input_dim, input_dim).to(device)
dummy_F = torch.zeros(input_dim).to(device)

train_mean_i = np.load('./results/train_mean_i.npy')
train_std_i = np.load('./results/train_std_i.npy')
model = DiscreteEnergyConservingModel(M_init=dummy_A, 
                              F_init=dummy_F, 
                              feature_means = train_mean_i, 
                              feature_stds = train_std_i, 
                              N=input_dim, hidden_nodes=hidden_dim, orthogonal_map='exp').to(device)
# 3. Load the saved weights
checkpoint_path = "./results/best_model.pth"
state_dict = torch.load(checkpoint_path, map_location=torch.device('cpu'))
#state_dict = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(state_dict)

# 3. Move model to device and SET TO EVAL MODE
model = model.to(device)
model.eval() # <--- Crucial for inference!

########## Noise covariance matrix
Sigma_matrix = np.load('./results/Sigma_matrix_identity.npy')

# Computing responses

In [5]:
# Fast code to compute statistical responses

def run_streamed_ensembles_discrete(
    long_trajectory,
    model,
    Sigma,
    N_steps,
    mean_i,
    std_i,
    delta=None,
    N_total=int(1e7),
    N_batch=int(1e4),
    base_seed=12345,
    use_gpu=True,
    dtype=torch.float32,
    verbose=True,
):
    # --------- device & reproducibility setup ----------
    device = torch.device('cuda' if (torch.cuda.is_available() and use_gpu) else 'cpu')
    model = model.to(device).eval()
    
    Sigma_t = torch.tensor(Sigma, dtype=dtype, device=device)
    mean_t = torch.tensor(mean_i, dtype=dtype, device=device)
    std_t  = torch.tensor(std_i, dtype=dtype, device=device)

    dim = long_trajectory.shape[1]
    
    C_mean_global = None
    C_var_global  = None
    P_mean_global = None
    P_var_global  = None
    N_done = 0

    batch_index = 0
    rng_global = np.random.default_rng(base_seed)

    while N_done < N_total:
        Nens = int(min(N_batch, N_total - N_done))

        # 1) Sample initial conditions
        idx = rng_global.integers(0, len(long_trajectory), size=Nens)
        x0_batch_np = long_trajectory[idx]
        
        x_ctl = torch.tensor(x0_batch_np, dtype=dtype, device=device)
        if delta is not None:
            # Note: delta should be in standardized units if x0_batch_np is standardized
            x_prt = x_ctl + torch.tensor(delta, dtype=dtype, device=device)
        else:
            x_prt = None

        # Accumulators for this batch
        C_mean_batch = np.zeros((N_steps, dim))
        C_var_batch  = np.zeros((N_steps, dim))
        P_mean_batch = np.zeros((N_steps, dim)) if delta is not None else None
        P_var_batch  = np.zeros((N_steps, dim)) if delta is not None else None

        batch_seed = base_seed + batch_index
        gen = torch.Generator(device=device).manual_seed(int(batch_seed))

        # 3) Time integration loop
        with torch.no_grad():
            for t in range(N_steps):
                # Stats: de-standardize
                x_ctl_destd = mean_t + x_ctl * std_t
                C_mean_batch[t] = x_ctl_destd.mean(dim=0).cpu().numpy()
                C_var_batch[t]  = x_ctl_destd.var(dim=0).cpu().numpy()

                if x_prt is not None:
                    x_prt_destd = mean_t + x_prt * std_t
                    P_mean_batch[t] = x_prt_destd.mean(dim=0).cpu().numpy()
                    P_var_batch[t]  = x_prt_destd.var(dim=0).cpu().numpy()

                # --- DISCRETE UPDATE STEP ---
                # Predict next deterministic state
                x_ctl_next = model(x_ctl) 
                
                # Generate paired noise
                wiener = torch.randn((Nens, dim), dtype=dtype, device=device, generator=gen)
                noise = wiener @ Sigma_t.T
                
                # Update control
                x_ctl = x_ctl_next + noise
                
                # Update perturbed (using same noise realization)
                if x_prt is not None:
                    x_prt_next = model(x_prt)
                    x_prt = x_prt_next + noise

        # 4) Streaming Stats Update (Chan's Formula)
        if C_mean_global is None:
            C_mean_global, C_var_global = C_mean_batch.copy(), C_var_batch.copy()
            if delta is not None:
                P_mean_global, P_var_global = P_mean_batch.copy(), P_var_batch.copy()
        else:
            n1, n2 = N_done, Nens
            
            # Control Update
            d_m = C_mean_batch - C_mean_global
            C_mean_global += d_m * (n2 / (n1 + n2))
            C_var_global = (n1*C_var_global + n2*C_var_batch + (n1*n2/(n1+n2))*(d_m**2)) / (n1+n2)
            
            # Perturbed Update
            if delta is not None:
                d_m_p = P_mean_batch - P_mean_global
                P_mean_global += d_m_p * (n2 / (n1 + n2))
                P_var_global = (n1*P_var_global + n2*P_var_batch + (n1*n2/(n1+n2))*(d_m_p**2)) / (n1+n2)

        N_done += Nens
        batch_index += 1
        if verbose:
            print(f"Processed {N_done}/{N_total}", end='\r')

    return C_mean_global, C_var_global, (P_mean_global, P_var_global) if delta is not None else (None, None)

# Impulse response functions

In [6]:
# Load the long trajectory, but keep only the first 10^7 points
long_trajectory = np.load('./results/x_t_emulated.npy')#[0:10**4]

long_trajectory_std = (long_trajectory-mean_i)/std_i
sigmas = np.std(long_trajectory_std,0)

In [8]:
# ==========================================
# Run Perturbation Experiments
# ==========================================
period_long = 4000
N_total = 10**6
N_batch = 10**5  # Adjust based on GPU memory (VRAM)
base_seed = 12345

# Ensure the model is in eval mode and on the right device
model.eval().to(device)

for i in range(6):
    # epsilon imposed on the i-th degree of freedom
    # Note: sigmas[i] is the std of the standardized trajectory (should be ~1.0)
    eps = 0.1 * sigmas[i] 
    delta = np.zeros(6)
    delta[i] = eps
    
    print(f"\n--- Running Perturbation on Variable {i+1} ---")
    print(f"Standardized perturbation delta: {delta}")

    # Use the DISCRETE version of the runner
    C_mean, C_var, (P_mean, P_var) = run_streamed_ensembles_discrete(
        long_trajectory=long_trajectory_std,
        model=model,
        Sigma=Sigma_matrix,        # Using the loaded Sigma_matrix
        N_steps=period_long,
        mean_i=mean_i,
        std_i=std_i,
        delta=delta,               
        N_total=N_total,
        N_batch=N_batch,
        base_seed=base_seed,
        use_gpu=True,              
        dtype=torch.float32,       
        verbose=True
    )

    # Calculate Response Function
    # We divide by the PHYSICAL epsilon (eps * std_i[i]) to get the sensitivity
    eps_phys = eps * std_i[i]
    R_mean = (P_mean - C_mean) / eps_phys
    R_var = (P_var - C_var) / eps_phys

    # Save results
    np.save(f'./results_responses_impulse/response_mean_x_{i+1}.npy', R_mean)
    np.save(f'./results_responses_impulse/response_var_x_{i+1}.npy', R_var)
    print(f"Saved response for variable {i+1}")


--- Running Perturbation on Variable 1 ---
Standardized perturbation delta: [0.0939449 0.        0.        0.        0.        0.       ]
Saved response for variable 1

--- Running Perturbation on Variable 2 ---
Standardized perturbation delta: [0.        0.0948912 0.        0.        0.        0.       ]
Saved response for variable 2

--- Running Perturbation on Variable 3 ---
Standardized perturbation delta: [0.         0.         0.09609681 0.         0.         0.        ]
Saved response for variable 3

--- Running Perturbation on Variable 4 ---
Standardized perturbation delta: [0.         0.         0.         0.10383132 0.         0.        ]
Saved response for variable 4

--- Running Perturbation on Variable 5 ---
Standardized perturbation delta: [0.        0.        0.        0.        0.1054344 0.       ]
Saved response for variable 5

--- Running Perturbation on Variable 6 ---
Standardized perturbation delta: [0.         0.         0.         0.         0.         0.10052617

NOTE: 

If I load 

i = 1

R_mean = np.load('./results_responses_impulse/response_mean_x_'+str(i)+'.npy')

R_var = np.load('./results_responses_impulse/response_var_x_'+str(i)+'.npy')

It means that I am looking at how the ensemble mean (i.e. R_mean) or variance (i.e. R_var) of $x_1(t), x_2(t), x_3(t), x_4(t), x_5(t), x_6(t)$  responds to an impulse perturbation imposed on variable $x_1(t=0)$

Example: R_mean[:,0] will quantify the response in ensemble mean of $x_1$ to an impulse perturbation on $x_1(t=0)$;

R_mean[:,2] will quantify the response in ensemble mean of $x_3$ to an impulse perturbation on $x_1(t=0)$;

Etc.

In [12]:
i = 1
R_mean = np.load('./results_responses_impulse/response_mean_x_'+str(i)+'.npy')
R_var = np.load('./results_responses_impulse/response_var_x_'+str(i)+'.npy')

In [ ]:
i = 2
plt.plot(R_mean[:,i],label = 'Discrete')
plt.legend()

In [ ]:
i = 2
plt.plot(R_var[:,i],label = 'Discrete')
plt.legend()